In [4]:
import pandas as pd

# ====== ตั้งค่า ======
INPUT_FILE = r"C:\Users\User\Downloads\ไฟล์งาน_pyton_68030916.csv"
OUTPUT_FILE = r"C:\Users\User\Downloads\wine_clean.csv"

OUTLIER_METHOD = "cap"   # "cap" = ปรับค่าสุดโต่งให้อยู่ในขอบเขต, "remove" = ลบแถว, "none" = ไม่ทำอะไร
IQR_FACTOR = 3.0         # ยิ่งมากยิ่งจับเฉพาะค่าสุดโต่งจริง ๆ (1.5 = มาตรฐาน)

# ====== 1) โหลดข้อมูล ======
df = pd.read_csv(INPUT_FILE, encoding="utf-8")
print(f"โหลดข้อมูล: {df.shape[0]} แถว, {df.shape[1]} คอลัมน์")

# ====== 2) ทำชื่อคอลัมน์ให้เป็นมาตรฐาน ======
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
num_cols = [c for c in df.columns if c != "type"]

# ====== 3) แปลงชนิดข้อมูล ======
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["type"] = df["type"].astype(str).str.strip().str.lower()

# ====== 4) จัดการค่าว่าง ======
missing = int(df[num_cols].isna().sum().sum())
print(f"ค่าว่าง (missing): {missing}")
if missing > 0:
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# ====== 5) ลบแถวซ้ำ ======
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"ลบแถวซ้ำ: {before - len(df)} แถว")

# ====== 6) ลบแถวที่ค่าเป็นไปไม่ได้ ======
non_ph = [c for c in num_cols if c != "ph"]
invalid = (
    (df[non_ph] < 0).any(axis=1)
    | (df["free_sulfur_dioxide"] > df["total_sulfur_dioxide"])
    | (~df["quality"].between(0, 10))
    | (~df["ph"].between(0, 14))
)
print(f"ลบแถวที่ค่าเป็นไปไม่ได้: {int(invalid.sum())} แถว")
df = df[~invalid].reset_index(drop=True)

# ====== 7) จัดการ outlier (IQR) ======
feature_cols = [c for c in num_cols if c != "quality"]
q1, q3 = df[feature_cols].quantile(0.25), df[feature_cols].quantile(0.75)
iqr = q3 - q1
low, high = q1 - IQR_FACTOR * iqr, q3 + IQR_FACTOR * iqr
is_out = (df[feature_cols] < low) | (df[feature_cols] > high)
print(f"Outlier ที่พบ: {int(is_out.sum().sum())} ค่า ใน {int(is_out.any(axis=1).sum())} แถว")

if OUTLIER_METHOD == "cap":
    df[feature_cols] = df[feature_cols].clip(lower=low, upper=high, axis=1)
elif OUTLIER_METHOD == "remove":
    df = df[~is_out.any(axis=1)].reset_index(drop=True)

# ====== 8) บันทึกผล ======
df["quality"] = df["quality"].astype(int)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"\nเสร็จแล้ว: {df.shape[0]} แถว, {df.shape[1]} คอลัมน์ -> {OUTPUT_FILE}")
df.head()


โหลดข้อมูล: 6497 แถว, 13 คอลัมน์
ค่าว่าง (missing): 0
ลบแถวซ้ำ: 1177 แถว
ลบแถวที่ค่าเป็นไปไม่ได้: 0 แถว
Outlier ที่พบ: 282 ค่า ใน 260 แถว

เสร็จแล้ว: 5320 แถว, 13 คอลัมน์ -> C:\Users\User\Downloads\wine_clean.csv


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5,red
